In [1]:
0+9

9

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
load_dotenv(override=True)

search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

sandbox=os.path.abspath("sandbox")
os.makedirs(sandbox,exist_ok=True)

model=ChatOpenAI(model="gpt-3.5-turbo")

research = create_deep_agent(
model=model, 
tools=[search],
system_prompt=("you are a reserch analyst. plan your work with todo tool"
"research with search tool and write your findings as a tidy markdown briefing to the file"),
backend = FilesystemBackend(root_dir=sandbox, virtual_mode=True),)


C:\Users\a810557\AppData\Local\Temp\ipykernel_26600\1342446611.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import GoogleSerperRun


In [3]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in the US: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""

result=research.invoke({"messages":[{"role":"user","content":brief}]})

print(result["messages"][-1].content)


I have updated the total number of public EV charging points in the US to over 250,000 in the charging.md file. The briefing is now complete. 

You can download the file [here](sandbox:/charging.md).


In [4]:
#Sub Agent

research_ev_instructions = """You research one electric vehicle using the search tool and 
return three concise facts that a fleet buyer would care about ,
such as price , range charging"""

overall_instructions = """you write comparison briefing for a company choosing electric vehicles 
for it sales fleet .For each vehicle, delegate the research to your vehicle-research sub-agent, 
then write a markdown comparison to a file,ending with a clear recommendations"""

research_subagent = {
    "name": "vehicle-researcher",
    "description":"Researches a single electric vehicle and returns short list of facts about it",
    "system_prompt": research_ev_instructions,
}

lead = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=overall_instructions,
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True)
)

mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""
result = lead.invoke({'messages':[{'role':"user","content":mission}]})
print(result['messages'][-1].content)

The comparison between the Tesla Model Y and the Ford Mustang Mach-E for the sales fleet has been summarized in the `fleet.md` file. Here is the content of the comparison:

```markdown
# Electric Vehicle Comparison for Sales Fleet

## Tesla Model Y
- Starting at an MSRP of $37,795
- EPA-estimated range of around 260 miles
- Key features include a 10.2" cluster display and 15.5" center display

## Ford Mustang Mach-E
- Key Types: The Tesla Model Y supports different types of keys, including phone keys, key cards, and key fobs. Key fobs require a CR2032 3V battery and are compatible with all Model 3 and Model Y vehicles. The retail price for a Model 3/Y Key Fob is $175.
- Key Limit: Model Y supports up to 19 keys at a time, with four of those keys being key fobs. When the limit of keys is reached, a key must be deleted before adding a new one.
- Key Accessories: Tesla vehicles, including Model Y, come with two Key Cards and one bifold wallet for easy entry, operation, and transport. Addi

In [5]:
%pip install python-pptx

Note: you may need to restart the kernel to use updated packages.


c:\Users\a810557\akash\LangchainUpdated\.venv\Scripts\python.exe: No module named pip


In [ ]:
from langchain_core.tools import tool
from pptx import Presentation

@tool
def create_slide(title: str, key_points: list[str], recommendataion: str) -> str:
    """Create a one-slide PowerPoint in the voltway research house style, saved as fleet.pptx"""
prs = Presentation('udemy/sandbox/SeniorDesignTitleSlide.pptx')

Presentation_Title = prs.placeholders[0]
Presentation_Subtitle = prs.placeholders[1]
Presentation_Title.text = 'This Is a Test'
Presentation_Subtitle.text = 'Is This Working?'

prs.save('SlideLayoutImportTest.pptx')

    Presentation(, key_points, recommendataion, os.path.join(sandbox,"fleet.pptx"))
    return "Saved the slide to /fleet.pptx"

slide_maker = { 
    "name": "slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck",
    "system_prompt":"You turn research recommendations into slides, following your fleet-slide skill",
    "tools":[create_slide],
    "skills":["/skills/"]
}

presenter = create_deep_agent(
    model=model,
    subagents=[slide_maker],
    system_prompt="You prepare research for presentation by delegating to your slide-maker sub-agent",
    backend=FilesystemBackend(root_dir=sandbox,virtual_mode=True)
)

result = presenter.invoke({'messages':[{"role":"user","content":"Read fleet.md and have a one-slide deck made of its recommendation"}]})
print(result['messages'][-1].content)


TypeError: Presentation() takes from 0 to 1 positional arguments but 4 were given

In [11]:
from slide_kit import build_deck


ModuleNotFoundError: No module named 'slide_kit'